# Toolthread — BFCL multi-turn eval (Kaggle)

Thin runner for Kaggle's free GPU (**2x T4, SM75**). T4 does not support sglang, so this notebook uses **vLLM** only and does **not** reinstall torch (keep Kaggle's CUDA-matched build).

Add `HF_TOKEN` and `WANDB_API_KEY` as Kaggle Secrets (Add-ons -> Secrets) before running. Turn on the GPU accelerator in the notebook settings.

In [ ]:
import os, subprocess
REPO = "https://github.com/aniketqxp/toolthread.git"
if not os.path.isdir("toolthread"):
    subprocess.run(["git", "clone", REPO], check=True)
os.chdir("toolthread")
print("cwd:", os.getcwd())

In [ ]:
# Secrets from Kaggle Secrets manager.
import os
from kaggle_secrets import UserSecretsClient
sec = UserSecretsClient()
os.environ["HF_TOKEN"] = sec.get_secret("HF_TOKEN")
os.environ["WANDB_API_KEY"] = sec.get_secret("WANDB_API_KEY")
os.environ["BFCL_PROJECT_ROOT"] = "/kaggle/working/bfcl_runs"
print("secrets loaded; BFCL_PROJECT_ROOT =", os.environ["BFCL_PROJECT_ROOT"])

In [ ]:
# Pinned stable layer:
!pip install -q -r requirements.txt
# Kaggle GPU is 2x T4 (SM75) -> vLLM only (sglang needs SM80+).
# Do NOT reinstall torch here; keep Kaggle's CUDA-matched build.
!pip install -q "bfcl-eval[oss_eval_vllm,wandb]==2026.3.23"
# Make src/ importable:
!pip install -q -e . --no-deps

In [ ]:
# Baseline candidate A (Qwen3-1.7B), full multi-turn, vLLM, single T4.
!python scripts/run_bfcl_eval.py \
  --model-config configs/model/qwen3-1.7b.yaml \
  --eval-config  configs/eval/bfcl_multi_turn.yaml \
  --backend vllm --num-gpus 1